# Flood Area Segmentation

This notebook contains a complete, production-ready pipeline for Binary Semantic Segmentation of flooded regions in aerial/drone images.

## Project Explanations (Simple Language)

1. **Architecture (U-Net + EfficientNet-B3)**:
   - Think of the model like an hourglass. The first half (the encoder, EfficientNet-B3) compresses the image to understand "what" is in it (features like water textures, edges). The second half (decoder) expands it back up to full size to understand "where" the water is. 
   - We use ImageNet pre-trained weights, which means the model has already learned basic shapes and textures from millions of everyday images, giving it a massive head start.

2. **Training Pipeline**:
   - We feed the model batches of images and their true masks.
   - The model makes a guess, checks how wrong it is (using a loss function), and adjusts its internal numbers (weights) to be more accurate next time using an optimizer (AdamW).
   - We use Mixed Precision (AMP), which uses smaller decimal numbers on the GPU to make training faster and use less memory without losing accuracy.
   - We also use Early Stopping: if the model stops getting better on unseen validation data, we stop training early so it doesn't just memorize the training data.

3. **Loss Functions (Dice + BCE)**:
   - **BCE (Binary Cross Entropy)**: Looks at every single pixel independently and asks, "Is this water or not?" It gives stable and steady learning.
   - **Dice Loss**: Looks at the whole predicted puddle of water and sees how well it overlaps with the actual puddle. This is great for when floods are small compared to dry land (class imbalance).
   - We combine them to get the best of both worlds: stable learning and perfect puddle overlap.

4. **Evaluation Metrics**:
   - **Dice Score**: Measures the exact overlap between our prediction and the truth (2 * Overlap / Total Area).
   - **IoU (Intersection over Union)**: Similar to Dice, just calculated slightly differently (Overlap / Union).
   - **Precision**: When the model says "This is water", how often is it actually right?
   - **Recall**: Out of all the actual water in the image, how much did the model manage to find?

5. **Inference Process**:
   - We take our best trained model, give it a brand new drone image it has never seen, and it outputs a black-and-white mask. We can then overlay this mask on the original image (e.g., highlighting it in blue) to visually see the flood area.


In [1]:
!pip install -q segmentation-models-pytorch albumentations opencv-python



[notice] A new release of pip available: 22.3.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import cv2
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from tqdm import tqdm

import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

# Set device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


c:\Users\Ashutosh Pandey\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [3]:
# Data Cleaning
def clean_data(data_dir="Data"):
    image_dir = os.path.join(data_dir, 'Image')
    mask_dir = os.path.join(data_dir, 'Mask')
    
    if not os.path.exists(image_dir) or not os.path.exists(mask_dir):
        print(f"Directories {image_dir} or {mask_dir} not found. Please ensure data is present.")
        return []

    images = os.listdir(image_dir)
    masks = os.listdir(mask_dir)
    
    image_stems = {Path(img).stem: img for img in images}
    mask_stems = {Path(mask).stem: mask for mask in masks}
    
    common_stems = set(image_stems.keys()).intersection(set(mask_stems.keys()))
    
    valid_pairs = []
    for stem in common_stems:
        valid_pairs.append({
            'Image': os.path.join(image_dir, image_stems[stem]),
            'Mask': os.path.join(mask_dir, mask_stems[stem])
        })
        
    print(f"Found {len(valid_pairs)} valid image-mask pairs out of {len(images)} images.")
    return valid_pairs

valid_data = clean_data()
df = pd.DataFrame(valid_data)
# Split into train/val (80/20)
train_df = df.sample(frac=0.8, random_state=42)
val_df = df.drop(train_df.index)
print(f"Train size: {len(train_df)}, Val size: {len(val_df)}")


Found 290 valid image-mask pairs out of 290 images.
Train size: 232, Val size: 58


In [4]:
# Dataset and Augmentations
class FloodDataset(Dataset):
    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_path = self.dataframe.loc[idx, 'Image']
        mask_path = self.dataframe.loc[idx, 'Mask']
        
        # Read image and mask
        image = cv2.imread(img_path)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        # Convert mask to binary (0 and 1)
        # Fix shape mismatch if any
        if image.shape[:2] != mask.shape[:2]:
            mask = cv2.resize(mask, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)
        mask = (mask > 0).astype(np.float32)
        
        if self.transform:
            augmented = self.transform(image=image, mask=mask)
            image = augmented['image']
            mask = augmented['mask']
            
        # Add channel dimension to mask [1, H, W]
        mask = mask.unsqueeze(0)
        
        return image, mask

# Transforms
train_transform = A.Compose([
    A.Resize(512, 512),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.RandomBrightnessContrast(p=0.2),
    A.RandomRain(brightness_coefficient=0.9, drop_length=15, blur_value=3, p=0.2),
    A.RandomFog(fog_coef_lower=0.2, fog_coef_upper=0.4, alpha_coef=0.08, p=0.2),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_transform = A.Compose([
    A.Resize(512, 512),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# DataLoaders
train_dataset = FloodDataset(train_df, transform=train_transform)
val_dataset = FloodDataset(val_df, transform=val_transform)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0, pin_memory=True)


In [5]:
# Model Initialization
# We use segmentation_models_pytorch for the U-Net with EfficientNet-B3 backbone
model = smp.Unet(
    encoder_name="efficientnet-b3", 
    encoder_weights="imagenet",     
    in_channels=3,                  
    classes=1,                      
    activation=None                 # We will use BCEWithLogitsLoss which includes Sigmoid
).to(device)

print("Model successfully loaded with ImageNet weights!")


Model successfully loaded with ImageNet weights!


In [6]:
# Metrics and Loss Functions
class DiceBCELoss(nn.Module):
    def __init__(self, weight=None, size_average=True):
        super(DiceBCELoss, self).__init__()
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, inputs, targets, smooth=1):
        # Apply sigmoid to outputs
        inputs_sigmoid = torch.sigmoid(inputs)
        
        # Flatten label and prediction tensors
        inputs_flat = inputs_sigmoid.view(-1)
        targets_flat = targets.view(-1)
        
        # Calculate BCE loss
        bce_loss = self.bce(inputs, targets)
        
        # Calculate Dice loss
        intersection = (inputs_flat * targets_flat).sum()
        dice_loss = 1 - (2.*intersection + smooth)/(inputs_flat.sum() + targets_flat.sum() + smooth)
        
        # Combine
        Dice_BCE = bce_loss + dice_loss
        return Dice_BCE

def calculate_metrics(pred, target, threshold=0.5):
    pred = (torch.sigmoid(pred) > threshold).float()
    
    pred_flat = pred.view(-1)
    target_flat = target.view(-1)
    
    tp = (pred_flat * target_flat).sum()
    fp = (pred_flat * (1 - target_flat)).sum()
    fn = ((1 - pred_flat) * target_flat).sum()
    
    smooth = 1e-6
    precision = tp / (tp + fp + smooth)
    recall = tp / (tp + fn + smooth)
    
    intersection = (pred_flat * target_flat).sum()
    union = pred_flat.sum() + target_flat.sum() - intersection
    iou = (intersection + smooth) / (union + smooth)
    
    dice = (2. * intersection + smooth) / (pred_flat.sum() + target_flat.sum() + smooth)
    
    return dice.item(), iou.item(), precision.item(), recall.item()


In [7]:
# Training Loop
def train_model(model, train_loader, val_loader, epochs=30):
    criterion = DiceBCELoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
    scaler = torch.cuda.amp.GradScaler() # For Mixed Precision
    
    best_dice = 0.0
    early_stopping_patience = 7
    epochs_no_improve = 0
    
    history = {'train_loss': [], 'val_loss': [], 'val_dice': [], 'val_iou': []}

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0
        
        # Use tqdm for progress bar
        train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs} [Train]")
        for images, masks in train_pbar:
            images, masks = images.to(device), masks.to(device)
            
            optimizer.zero_grad()
            
            with torch.cuda.amp.autocast():
                outputs = model(images)
                loss = criterion(outputs, masks)
                
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            train_loss += loss.item() * images.size(0)
            train_pbar.set_postfix({'loss': loss.item()})
            
        train_loss = train_loss / len(train_loader.dataset)
        
        # Validation Phase
        model.eval()
        val_loss = 0.0
        val_dice, val_iou, val_precision, val_recall = 0.0, 0.0, 0.0, 0.0
        
        val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1}/{epochs} [Val]")
        with torch.no_grad():
            for images, masks in val_pbar:
                images, masks = images.to(device), masks.to(device)
                
                outputs = model(images)
                loss = criterion(outputs, masks)
                val_loss += loss.item() * images.size(0)
                
                # Metrics
                dice, iou, prec, rec = calculate_metrics(outputs, masks)
                val_dice += dice * images.size(0)
                val_iou += iou * images.size(0)
                val_precision += prec * images.size(0)
                val_recall += rec * images.size(0)
                
        val_loss = val_loss / len(val_loader.dataset)
        val_dice = val_dice / len(val_loader.dataset)
        val_iou = val_iou / len(val_loader.dataset)
        val_precision = val_precision / len(val_loader.dataset)
        val_recall = val_recall / len(val_loader.dataset)
        
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['val_dice'].append(val_dice)
        history['val_iou'].append(val_iou)
        
        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | "
              f"Val Dice: {val_dice:.4f} | Val IoU: {val_iou:.4f} | Prec: {val_precision:.4f} | Rec: {val_recall:.4f}")
        
        scheduler.step(val_dice)
        
        # Checkpoint Saving
        if val_dice > best_dice:
            best_dice = val_dice
            torch.save(model.state_dict(), 'best_model.pth')
            print(f"-> Saved new best model with Dice Score: {best_dice:.4f}")
            epochs_no_improve = 0
        else:
            epochs_no_improve += 1
            if epochs_no_improve >= early_stopping_patience:
                print(f"Early stopping triggered after {epoch+1} epochs.")
                break
                
    return history


In [8]:
# Run Training
if len(train_loader) > 0:
    history = train_model(model, train_loader, val_loader, epochs=30)
else:
    print("No data found to train on. Add images to Data/Image and Data/Mask to start training!")


C:\Users\Ashutosh Pandey\AppData\Local\Temp\ipykernel_35488\2317924591.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() # For Mixed Precision
Epoch 1/30 [Train]:   0%|          | 0/29 [00:00<?, ?it/s]C:\Users\Ashutosh Pandey\AppData\Local\Temp\ipykernel_35488\2317924591.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/30 [Val]: 100%|██████████| 8/8 [00:02<00:00,  3.02it/s]


Epoch 1/30 | Train Loss: 1.1195 | Val Loss: 0.9636 | Val Dice: 0.7236 | Val IoU: 0.5681 | Prec: 0.6483 | Rec: 0.8218
-> Saved new best model with Dice Score: 0.7236


Epoch 2/30 [Val]: 100%|██████████| 8/8 [00:02<00:00,  3.28it/s]


Epoch 2/30 | Train Loss: 0.8502 | Val Loss: 0.7884 | Val Dice: 0.7967 | Val IoU: 0.6629 | Prec: 0.7137 | Rec: 0.9031
-> Saved new best model with Dice Score: 0.7967


Epoch 3/30 [Val]: 100%|██████████| 8/8 [00:02<00:00,  3.25it/s]


...

Epoch 28/30 | Train Loss: 0.2557 | Val Loss: 0.3405 | Val Dice: 0.9058 | Val IoU: 0.8279 | Prec: 0.9139 | Rec: 0.8983


Epoch 29/30 [Val]: 100%|██████████| 8/8 [00:02<00:00,  3.37it/s]


Epoch 29/30 | Train Loss: 0.2637 | Val Loss: 0.3401 | Val Dice: 0.9058 | Val IoU: 0.8279 | Prec: 0.9155 | Rec: 0.8967


Epoch 30/30 [Val]: 100%|██████████| 8/8 [00:02<00:00,  3.34it/s]

Epoch 30/30 | Train Loss: 0.2535 | Val Loss: 0.3403 | Val Dice: 0.9059 | Val IoU: 0.8283 | Prec: 0.9127 | Rec: 0.8997
Early stopping triggered after 30 epochs.


In [9]:
# Inference and Visualization
def predict_and_visualize(model_path, image_path, device='cpu'):
    if not os.path.exists(image_path) or not os.path.exists(model_path):
        print("Model or Image path does not exist for inference.")
        return
        
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    
    # Read and transform image
    image = cv2.imread(image_path)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    
    transform = A.Compose([
        A.Resize(512, 512),
        A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
        ToTensorV2()
    ])
    
    tensor_img = transform(image=image_rgb)['image'].unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(tensor_img)
        pred_mask = torch.sigmoid(output).squeeze().cpu().numpy()
        pred_mask = (pred_mask > 0.5).astype(np.uint8)
        
    # Resize mask back to original image size
    pred_mask_resized = cv2.resize(pred_mask, (image.shape[1], image.shape[0]), interpolation=cv2.INTER_NEAREST)
    
    # Create an overlay (highlight flood in blue)
    overlay = image_rgb.copy()
    overlay[pred_mask_resized == 1] = overlay[pred_mask_resized == 1] * 0.5 + np.array([0, 0, 255]) * 0.5
    
    # Save predicted mask
    cv2.imwrite('predicted_mask.png', pred_mask_resized * 255)
    
    # Visualize
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(image_rgb)
    axes[0].set_title("Original Image")
    axes[0].axis('off')
    
    axes[1].imshow(pred_mask_resized, cmap='gray')
    axes[1].set_title("Predicted Mask")
    axes[1].axis('off')
    
    axes[2].imshow(overlay.astype(np.uint8))
    axes[2].set_title("Overlay")
    axes[2].axis('off')
    
    plt.show()

# Example usage (uncomment and replace with your test image path when ready):
# predict_and_visualize('best_model.pth', 'Data/Image/0.jpg', device)


---
# 🚀 Model Extensions & Comparisons

In this section, we extend our project to train two additional models for comparison:
1. **U-Net with ResNet34 encoder**
2. **DeepLabV3+ with ResNet50 encoder**

First, let's build an evaluation function and save the results of our existing `U-Net + EfficientNet-B3` model.

In [10]:
import time
import shutil
import os
import pandas as pd
from IPython.display import display

# Dictionary to store metrics for our final table
comparison_results = {}

def evaluate_model(model_to_eval, loader):
    """Evaluates the model on the entire validation loader and returns averaged metrics."""
    model_to_eval.eval()
    val_dice, val_iou, val_prec, val_rec = 0, 0, 0, 0
    with torch.no_grad():
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)
            outputs = model_to_eval(images)
            dice, iou, prec, rec = calculate_metrics(outputs, masks)
            val_dice += dice * images.size(0)
            val_iou += iou * images.size(0)
            val_prec += prec * images.size(0)
            val_rec += rec * images.size(0)
            
    n = len(loader.dataset)
    return val_dice/n, val_iou/n, val_prec/n, val_rec/n

# Rename and evaluate the original model so it's not overwritten
if os.path.exists('best_model.pth'):
    shutil.copy('best_model.pth', 'best_efficientnet_b3.pth')
    model.load_state_dict(torch.load('best_efficientnet_b3.pth', map_location=device))
    
    print("Evaluating U-Net + EfficientNet-B3...")
    dice, iou, prec, rec = evaluate_model(model, val_loader)
    
    comparison_results['U-Net + EfficientNet-B3'] = {
        'Dice Score': dice,
        'IoU': iou,
        'Precision': prec,
        'Recall': rec,
        'Training Time (s)': 'N/A (Already Trained)'
    }
    print(f'-> Dice: {dice:.4f}, IoU: {iou:.4f}, Precision: {prec:.4f}, Recall: {rec:.4f}')
else:
    print('best_model.pth not found. Train the first model above before running this!')


Evaluating U-Net + EfficientNet-B3...
-> Dice: 0.9064, IoU: 0.8290, Precision: 0.9097, Recall: 0.9035


---
## Model 2: U-Net + ResNet34 Encoder
We initialize a new U-Net using ResNet34 and ImageNet weights.

In [11]:
# Initialize Model 2
model_resnet34 = smp.Unet(
    encoder_name='resnet34', 
    encoder_weights='imagenet', 
    in_channels=3, 
    classes=1, 
    activation=None
).to(device)

print('U-Net + ResNet34 loaded successfully!')


U-Net + ResNet34 loaded successfully!


In [12]:
# Train Model 2
print('Starting training for U-Net + ResNet34...')
start_time = time.time()

history_resnet34 = train_model(model_resnet34, train_loader, val_loader, epochs=30)

end_time = time.time()
training_time_resnet34 = end_time - start_time
print(f'Training completed in {training_time_resnet34/60:.2f} minutes.')


C:\Users\Ashutosh Pandey\AppData\Local\Temp\ipykernel_35488\2317924591.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() # For Mixed Precision


Starting training for U-Net + ResNet34...


Epoch 1/30 [Train]:   0%|          | 0/29 [00:00<?, ?it/s]C:\Users\Ashutosh Pandey\AppData\Local\Temp\ipykernel_35488\2317924591.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/30 [Val]: 100%|██████████| 8/8 [00:02<00:00,  3.69it/s]


Epoch 1/30 | Train Loss: 0.9538 | Val Loss: 0.6983 | Val Dice: 0.8184 | Val IoU: 0.6933 | Prec: 0.8655 | Rec: 0.7777
-> Saved new best model with Dice Score: 0.8184


Epoch 2/30 [Val]: 100%|██████████| 8/8 [00:02<00:00,  3.36it/s]


...

Epoch 24/30 | Train Loss: 0.2700 | Val Loss: 0.3552 | Val Dice: 0.9044 | Val IoU: 0.8257 | Prec: 0.9153 | Rec: 0.8947


Epoch 25/30 [Val]: 100%|██████████| 8/8 [00:02<00:00,  3.79it/s]


Epoch 25/30 | Train Loss: 0.2612 | Val Loss: 0.3609 | Val Dice: 0.9029 | Val IoU: 0.8233 | Prec: 0.9233 | Rec: 0.8843


Epoch 26/30 [Val]: 100%|██████████| 8/8 [00:02<00:00,  3.69it/s]

Epoch 26/30 | Train Loss: 0.2569 | Val Loss: 0.3531 | Val Dice: 0.9049 | Val IoU: 0.8265 | Prec: 0.9173 | Rec: 0.8935
Early stopping triggered after 26 epochs.
Training completed in 9.53 minutes.


In [13]:
# Evaluate Model 2
if os.path.exists('best_model.pth'):
    shutil.copy('best_model.pth', 'best_resnet34.pth') # Rename checkpoint

model_resnet34.load_state_dict(torch.load('best_resnet34.pth', map_location=device))
dice, iou, prec, rec = evaluate_model(model_resnet34, val_loader)

comparison_results['U-Net + ResNet34'] = {
    'Dice Score': dice,
    'IoU': iou,
    'Precision': prec,
    'Recall': rec,
    'Training Time (s)': round(training_time_resnet34, 2)
}
print(f'U-Net + ResNet34 -> Dice: {dice:.4f}, IoU: {iou:.4f}')


U-Net + ResNet34 -> Dice: 0.9060, IoU: 0.8283


In [14]:
# Inference Model 2
print("Visualizing Inference for U-Net + ResNet34...")
# Uncomment the line below and change the path to visualize
# predict_and_visualize('best_resnet34.pth', 'Data/Image/0.jpg', device)


Visualizing Inference for U-Net + ResNet34...


---
## Model 3: DeepLabV3+ 
DeepLabV3+ uses Atrous Spatial Pyramid Pooling (ASPP). We use ResNet50 as the backbone.

In [15]:
# Initialize Model 3
model_deeplab = smp.DeepLabV3Plus(
    encoder_name='resnet50', 
    encoder_weights='imagenet', 
    in_channels=3, 
    classes=1, 
    activation=None
).to(device)

print('DeepLabV3+ loaded successfully!')


DeepLabV3+ loaded successfully!


In [16]:
# Train Model 3
print('Starting training for DeepLabV3+...')
start_time = time.time()

history_deeplab = train_model(model_deeplab, train_loader, val_loader, epochs=30)

end_time = time.time()
training_time_deeplab = end_time - start_time
print(f'Training completed in {training_time_deeplab/60:.2f} minutes.')


C:\Users\Ashutosh Pandey\AppData\Local\Temp\ipykernel_35488\2317924591.py:6: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() # For Mixed Precision


Starting training for DeepLabV3+...


Epoch 1/30 [Train]:   0%|          | 0/29 [00:00<?, ?it/s]C:\Users\Ashutosh Pandey\AppData\Local\Temp\ipykernel_35488\2317924591.py:25: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
Epoch 1/30 [Val]: 100%|██████████| 8/8 [00:02<00:00,  3.31it/s]


Epoch 1/30 | Train Loss: 0.8196 | Val Loss: 0.8314 | Val Dice: 0.8227 | Val IoU: 0.6995 | Prec: 0.8059 | Rec: 0.8423
-> Saved new best model with Dice Score: 0.8227


Epoch 2/30 [Val]: 100%|██████████| 8/8 [00:02<00:00,  3.53it/s]


...

Epoch 28/30 | Train Loss: 0.2252 | Val Loss: 0.3659 | Val Dice: 0.9034 | Val IoU: 0.8240 | Prec: 0.9082 | Rec: 0.8990
-> Saved new best model with Dice Score: 0.9034


Epoch 29/30 [Val]: 100%|██████████| 8/8 [00:02<00:00,  3.65it/s]


Epoch 29/30 | Train Loss: 0.2143 | Val Loss: 0.3605 | Val Dice: 0.9025 | Val IoU: 0.8225 | Prec: 0.9140 | Rec: 0.8917


Epoch 30/30 [Val]: 100%|██████████| 8/8 [00:02<00:00,  3.59it/s]

Epoch 30/30 | Train Loss: 0.2122 | Val Loss: 0.3598 | Val Dice: 0.9009 | Val IoU: 0.8198 | Prec: 0.9142 | Rec: 0.8886
Training completed in 6.36 minutes.


In [17]:
# Evaluate Model 3
if os.path.exists('best_model.pth'):
    shutil.copy('best_model.pth', 'best_deeplab.pth')

model_deeplab.load_state_dict(torch.load('best_deeplab.pth', map_location=device))
dice, iou, prec, rec = evaluate_model(model_deeplab, val_loader)

comparison_results['DeepLabV3+'] = {
    'Dice Score': dice,
    'IoU': iou,
    'Precision': prec,
    'Recall': rec,
    'Training Time (s)': round(training_time_deeplab, 2)
}
print(f'DeepLabV3+ -> Dice: {dice:.4f}, IoU: {iou:.4f}')


DeepLabV3+ -> Dice: 0.9034, IoU: 0.8240


In [18]:
# Inference Model 3
print("Visualizing Inference for DeepLabV3+...")
# predict_and_visualize('best_deeplab.pth', 'Data/Image/0.jpg', device)


Visualizing Inference for DeepLabV3+...


---
## Final Comparison and Best Model Selection
We generate a tabular comparison of all three models and programmatically pick the best one.

In [19]:
# Generate the Table
comparison_df = pd.DataFrame(comparison_results).T
comparison_df = comparison_df[['Dice Score', 'IoU', 'Precision', 'Recall', 'Training Time (s)']]

print("\n===================== MODEL COMPARISON =====================")
display(comparison_df)
print("==========================================================\n")

# Identify the Best Model
best_model_name = comparison_df['Dice Score'].idxmax()
best_dice = comparison_df.loc[best_model_name, 'Dice Score']

print(f"🏆 THE BEST PERFORMING MODEL IS: ** {best_model_name} ** 🏆")
print(f"🥇 It achieved the highest Validation Dice Score: {best_dice:.4f}")



===================== MODEL COMPARISON =====================


,Dice Score,IoU,Precision,Recall,Training Time (s)
U-Net + EfficientNet-B3,0.906404,0.829017,0.909738,0.903534,N/A (Already Trained)
U-Net + ResNet34,0.905963,0.828339,0.919247,0.894102,571.94
DeepLabV3+,0.90341,0.82396,0.908248,0.899048,381.83



🏆 THE BEST PERFORMING MODEL IS: ** U-Net + EfficientNet-B3 ** 🏆
🥇 It achieved the highest Validation Dice Score: 0.9064
